In [22]:
#Import all necessary libraries
from pathlib import Path
import pandas as pd
from sklearn.model_selection import StratifiedGroupKFold
from hashlib import sha256
import json, gc, zipfile
import numpy as np
import tensorflow as tf
from sklearn.metrics import (classification_report, confusion_matrix,
    accuracy_score, f1_score, balanced_accuracy_score, roc_auc_score,
    average_precision_score)
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.metrics import (
    confusion_matrix,
    ConfusionMatrixDisplay,
    roc_curve,
    auc
)

In [2]:
MAPPING = {
    **dict.fromkeys(['NV', 'BKL', 'DF', 'VASC'], 'benign'),
    **dict.fromkeys(['MEL', 'BCC', 'SCC', 'AK'], 'malignant')
}

fingerprint_path = Path.cwd().parent / "artifacts" / "dataset_fingerprint.json"

with open(fingerprint_path) as f:
    fingerprint_data = json.load(f)
fingerprint = fingerprint_data["fingerprint"]

In [3]:
data_path = Path.cwd().parent / "artifacts" / "isic2019_binary_split.csv"
out = Path.cwd().parent /'artifacts'
out.mkdir(parents=True, exist_ok=True)
df = pd.read_csv(data_path)
print(df.head())
print(df.columns)
print(df["split"].value_counts())
print(df.groupby(["split", "label"]).size().unstack(fill_value=0))

       image_id                                               path      label  \
0  ISIC_0000001   e:\Nová složka\input\Test\nevus\ISIC_0000001.jpg     benign   
1  ISIC_0000002  e:\Nová složka\input\Test\melanoma\ISIC_000000...  malignant   
2  ISIC_0000006   e:\Nová složka\input\Test\nevus\ISIC_0000006.jpg     benign   
3  ISIC_0000007   e:\Nová složka\input\Test\nevus\ISIC_0000007.jpg     benign   
4  ISIC_0000008   e:\Nová složka\input\Test\nevus\ISIC_0000008.jpg     benign   

  lesion_id               group  split original_label  target  
0       NaN  image:ISIC_0000001  train             NV       0  
1       NaN  image:ISIC_0000002  train            MEL       1  
2       NaN  image:ISIC_0000006  train             NV       0  
3       NaN  image:ISIC_0000007  train             NV       0  
4       NaN  image:ISIC_0000008  train             NV       0  
Index(['image_id', 'path', 'label', 'lesion_id', 'group', 'split',
       'original_label', 'target'],
      dtype='object')
spli

In [4]:
#Prepare training & validation dataframes
train = df[df["split"] == "train"].copy()
val   = df[df["split"] == "val"].copy()
test  = df[df["split"] == "test"].copy()

assert len(train) > 0
assert len(val) > 0
assert len(test) > 0

In [5]:
# Calculate class weights
class_weights = {
    int(k): len(train) / (2 * int(v))
    for k, v in train.target.value_counts().items()
}

print(pd.crosstab(df.label, df.split, margins=True))
print('Class weights:', class_weights)

split      test  train  val   All
label                            
benign      177    538  179   894
malignant   175    518  174   867
All         352   1056  353  1761
Class weights: {0: 0.9814126394052045, 1: 1.0193050193050193}


In [6]:
#Check available hardware (GPU vs CPU)
#DISCUSS ON THURSDAY
#assert tf.config.list_physical_devices('GPU'), 'Enable GPU before training.'
gpus = tf.config.list_physical_devices('GPU')

if gpus:
    print("GPU detected:", gpus)
else:
    print("No compatible GPU detected. Running on CPU.")

print("TensorFlow devices:", tf.config.list_physical_devices())

No compatible GPU detected. Running on CPU.
TensorFlow devices: [PhysicalDevice(name='/physical_device:CPU:0', device_type='CPU')]


In [7]:
#Define image loading
def load_image(path, target):
    x = tf.io.decode_image(tf.io.read_file(path), channels=3,
                           expand_animations=False)
    x.set_shape([None, None, 3])
    return tf.cast(tf.image.resize(x, [128,128]), tf.float32)/255., target

In [8]:
#Define TensorFlow dataset function
def dataset(part, seed, shuffle):
    ds = tf.data.Dataset.from_tensor_slices((part.path.astype(str).to_numpy(),
                                            part.target.to_numpy('int32')))
    ds = ds.map(load_image, num_parallel_calls=tf.data.AUTOTUNE)
    if shuffle:
        ds = ds.shuffle(len(part), seed=seed, reshuffle_each_iteration=True)
    opts = tf.data.Options()
    opts.experimental_deterministic = True
    return ds.with_options(opts).batch(32).prefetch(tf.data.AUTOTUNE)

In [9]:
#Separate train & val datasets for grid search
train_ds = dataset(train, seed=42, shuffle=True)
val_ds = dataset(val, seed=42, shuffle=False)

In [10]:
def build_cnn(filters, kernel_size, dense_units, dropout_rate):

    model = tf.keras.Sequential([
        tf.keras.layers.Input(shape=(128, 128,3)),

        tf.keras.layers.Conv2D(filters[0], kernel_size, activation='relu', padding='same'),
        tf.keras.layers.MaxPooling2D(),

        tf.keras.layers.Conv2D(filters[1], kernel_size, activation='relu', padding='same'),
        tf.keras.layers.MaxPooling2D(),

        tf.keras.layers.Conv2D(filters[2], kernel_size, activation='relu', padding='same'),
        tf.keras.layers.MaxPooling2D(),

        tf.keras.layers.Conv2D(filters[3], kernel_size, activation='relu', padding='same'),
        tf.keras.layers.MaxPooling2D(),

        tf.keras.layers.Flatten(),

        tf.keras.layers.Dense(dense_units, activation='relu'),
        tf.keras.layers.Dropout(dropout_rate),

        tf.keras.layers.Dense(1, activation='sigmoid')
    ])

    return model

In [11]:
#Grid search
def hyperparameter_search_cnn(filters, kernel_sizes,
                              dense_units, dropout_rates,
                              learning_rates):

    val_acc_list = []

    for i in filters:
        for j in kernel_sizes:
            for k in dense_units:
                for z in dropout_rates:
                    for lr in learning_rates:

                        tf.keras.backend.clear_session()
                        tf.keras.utils.set_random_seed(42)

                        model = build_cnn(
                            filters=i,
                            kernel_size=j,
                            dense_units=k,
                            dropout_rate=z
                        )

                        model.compile(
                            optimizer=tf.keras.optimizers.Adam(
                                learning_rate=lr
                            ),
                            loss='binary_crossentropy',
                            metrics=['accuracy']
                        )

                        history = model.fit(
                            train_ds,
                            validation_data=val_ds,
                            epochs=25,
                            class_weight=class_weights,
                            callbacks=[
                                tf.keras.callbacks.EarlyStopping(
                                    monitor='val_loss',
                                    patience=5,
                                    restore_best_weights=True
                                )
                            ],
                            verbose=0
                        )

                        y_val_pred = (
                            model.predict(val_ds, verbose=0).reshape(-1) >= 0.5
                        ).astype(int)

                        y_val = val.target.to_numpy()

                        output = {
                            "filters": str(i),
                            "kernel_size": j,
                            "dense_units": k,
                            "dropout_rate": z,
                            "learning_rate": lr,
                            "val_acc": accuracy_score(y_val, y_val_pred),
                            "val_f1": f1_score(
                                y_val, y_val_pred,
                                average='macro',
                                zero_division=0
                            ),
                            "val_auc": roc_auc_score(
                                y_val,
                                model.predict(val_ds, verbose=0).reshape(-1)
                            ),
                            "val_loss": min(history.history['val_loss'])
                        }

                        val_acc_list.append(output)

    df_output = pd.DataFrame(val_acc_list)
    df_output["Model"] = range(1, df_output.shape[0] + 1)

    return df_output

In [12]:
filters = [
    (32, 64, 128, 256),
    (32, 64, 128, 128)
]

kernel_sizes = [3, 5]

dense_units = [64, 128]

dropout_rates = [0.25, 0.35, 0.45]

learning_rates = [0.001, 0.0001]

In [13]:
df_cnn = hyperparameter_search_cnn(
    filters,
    kernel_sizes,
    dense_units,
    dropout_rates,
    learning_rates
)

print(df_cnn.sort_values("val_acc", ascending=False))


               filters  kernel_size  dense_units  dropout_rate  learning_rate  \
7   (32, 64, 128, 256)            3          128          0.25         0.0001   
1   (32, 64, 128, 256)            3           64          0.25         0.0001   
31  (32, 64, 128, 128)            3          128          0.25         0.0001   
13  (32, 64, 128, 256)            5           64          0.25         0.0001   
3   (32, 64, 128, 256)            3           64          0.35         0.0001   
9   (32, 64, 128, 256)            3          128          0.35         0.0001   
27  (32, 64, 128, 128)            3           64          0.35         0.0001   
5   (32, 64, 128, 256)            3           64          0.45         0.0001   
19  (32, 64, 128, 256)            5          128          0.25         0.0001   
23  (32, 64, 128, 256)            5          128          0.45         0.0001   
29  (32, 64, 128, 128)            3           64          0.45         0.0001   
25  (32, 64, 128, 128)     

In [14]:
best_model = df_cnn.loc[df_cnn["val_loss"].idxmin()]
print(best_model)

filters          (32, 64, 128, 256)
kernel_size                       5
dense_units                     128
dropout_rate                   0.45
learning_rate                0.0001
val_acc                    0.739377
val_f1                     0.738451
val_auc                    0.830251
val_loss                   0.502692
Model                            24
Name: 23, dtype: object


In [23]:
best_filters = (32, 64, 128, 256)
best_kernel_size = 5
best_dense_units = 128
best_dropout_rate = 0.45
best_learning_rate = 0.0001

#Run the model 
results = []
for seed in [42,43,44]:
    tf.keras.backend.clear_session()
    tf.keras.utils.set_random_seed(seed)
    model = build_cnn(
        filters=best_filters,
        kernel_size=best_kernel_size,
        dense_units=best_dense_units,
        dropout_rate=best_dropout_rate
    )
    init_hash = sha256(b''.join(w.tobytes() for w in model.get_weights())).hexdigest()
    # Preserve initial arrays for future paired binary augmentation comparisons.
    np.savez_compressed(out/f'initial_weights_seed{seed}.npz', *model.get_weights())
    model.compile(optimizer=tf.keras.optimizers.Adam(
        learning_rate=best_learning_rate),
    loss='binary_crossentropy',
    metrics=['accuracy', tf.keras.metrics.AUC(name='auc')])
    run = f'binary_no_aug_seed{seed}'
    checkpoint = out/f'{run}.keras'
    train_ds, val_ds = dataset(train, seed, True), dataset(val, seed, False)
    history = model.fit(train_ds, validation_data=val_ds, epochs=25,
        class_weight=class_weights, callbacks=[
            tf.keras.callbacks.ModelCheckpoint(str(checkpoint),
                monitor='val_loss',save_best_only=True),
            tf.keras.callbacks.EarlyStopping(monitor='val_loss',patience=5,
                                             restore_best_weights=True)], verbose=2)
    best = tf.keras.models.load_model(checkpoint)
    probability = best.predict(val_ds,verbose=0).reshape(-1)
    y = val.target.to_numpy()
    pred = (probability >= .5).astype(int)
    tn,fp,fn,tp = confusion_matrix(y,pred,labels=[0,1]).ravel()
    row = dict(seed=seed,condition='binary_no_aug',
        epochs_trained=len(history.history['loss']),
        selected_epoch=int(np.argmin(history.history['val_loss'])+1),
        validation_loss=float(min(history.history['val_loss'])),
        accuracy=float(accuracy_score(y,pred)),
        macro_f1=float(f1_score(y,pred,labels=[0,1],average='macro',zero_division=0)),
        balanced_accuracy=float(balanced_accuracy_score(y,pred)),
        malignant_sensitivity=float(tp/(tp+fn)), specificity=float(tn/(tn+fp)),
        roc_auc=float(roc_auc_score(y,probability)),
        average_precision=float(average_precision_score(y,probability)),
        threshold=.5,initial_weights_sha256=init_hash,
        split_fingerprint=fingerprint,test_evaluated=False)
    report=classification_report(y,pred,labels=[0,1],
        target_names=['benign','malignant'],output_dict=True,zero_division=0)
    (out/f'{run}_metrics.json').write_text(json.dumps(
        dict(**row,classification_report=report),indent=2))
    pd.DataFrame(history.history).assign(
        epoch=lambda d: np.arange(1,len(d)+1)).to_csv(out/f'{run}_history.csv',index=False)
    predictions=val[['image_id','original_label','label','group']].copy()
    predictions['malignant_probability']=probability
    predictions['prediction']=pred
    predictions.to_csv(out/f'{run}_validation_predictions.csv',index=False)
    pd.DataFrame(confusion_matrix(y,pred,labels=[0,1]),
        index=['benign','malignant'],columns=['benign','malignant']).to_csv(
            out/f'{run}_confusion_matrix.csv')
    # Save confusion matrix as PNG
    cm = confusion_matrix(y, pred, labels=[0, 1])

    plt.figure(figsize=(7, 6))
    sns.heatmap(
        cm,
        annot=True,
        fmt='d',
        cmap='Blues',
        xticklabels=['Benign', 'Malignant'],
        yticklabels=['Benign', 'Malignant'],
        annot_kws={'size': 16}
    )

    plt.xlabel('Predicted label', fontsize=12)
    plt.ylabel('True label', fontsize=12)
    plt.title(f'Confusion Matrix - {run}', fontsize=14)
    plt.tight_layout()

    plt.savefig(
        out/f'{run}_confusion_matrix.png',
        dpi=300,
        bbox_inches='tight'
    )
    plt.close()
    # Generate ROC curve
    fpr, tpr, thresholds = roc_curve(y, probability)
    roc_auc = auc(fpr, tpr)

    plt.figure(figsize=(7, 6))

    plt.plot(
        fpr,
        tpr,
        color='darkorange',
        lw=2,
        label=f'ROC curve (AUC = {roc_auc:.3f})'
    )

    plt.plot(
        [0, 1],
        [0, 1],
        color='navy',
        lw=2,
        linestyle='--',
        label='Random classifier'
    )

    plt.xlim([0.0, 1.0])
    plt.ylim([0.0, 1.05])

    plt.xlabel('False Positive Rate', fontsize=12)
    plt.ylabel('True Positive Rate', fontsize=12)
    plt.title(f'ROC Curve - {run}', fontsize=14)

    plt.legend(loc='lower right')
    plt.grid(alpha=0.3)
    plt.tight_layout()

    plt.savefig(
        out/f'{run}_ROC_curve.png',
        dpi=300,
        bbox_inches='tight'
    )
    plt.close()
    
    results.append(row)
    pd.DataFrame(results).to_csv(out/'binary_baseline_results.csv',index=False)
    # Checkpoint each seed's complete evidence before moving to the next run.
    
    # Archive lightweight experimental evidence
    archive_path = out / f'{run}_artifacts.zip'

    files_to_archive = [
        out / f'{run}_metrics.json',
        out / f'{run}_history.csv',
        out / f'{run}_validation_predictions.csv',
        out / f'{run}_confusion_matrix.csv',
        out / f'{run}_confusion_matrix.png',
        out / f'{run}_ROC_curve.png',
        out / 'isic2019_binary_split.csv',
        out / f'initial_weights_seed{seed}.npz'
    ]

    with zipfile.ZipFile(
        archive_path,
        'w',
        compression=zipfile.ZIP_DEFLATED,
        compresslevel=6
    ) as archive:

        for p in files_to_archive:
            if p.exists():
                archive.write(p, p.name)
            else:
                print(f'Warning: missing file {p}')

    print(f'Archive saved: {archive_path}')
    print(f'Archive size: {archive_path.stat().st_size / (1024**2):.2f} MB')

    # Model and initial weights remain saved separately
    print(f'Model saved: {checkpoint}')
    print(f'Initial weights saved: initial_weights_seed{seed}.npz')

    print('BINARY_RUN_COMPLETE', json.dumps(row), flush=True)

    del model, best, train_ds, val_ds
    gc.collect()

Epoch 1/25
33/33 - 25s - 764ms/step - accuracy: 0.5142 - auc: 0.4976 - loss: 0.6974 - val_accuracy: 0.4929 - val_auc: 0.6306 - val_loss: 0.6934
Epoch 2/25
33/33 - 8s - 239ms/step - accuracy: 0.4915 - auc: 0.5065 - loss: 0.6936 - val_accuracy: 0.5071 - val_auc: 0.5112 - val_loss: 0.6929
Epoch 3/25
33/33 - 8s - 247ms/step - accuracy: 0.5114 - auc: 0.5139 - loss: 0.6932 - val_accuracy: 0.5042 - val_auc: 0.5319 - val_loss: 0.6927
Epoch 4/25
33/33 - 8s - 250ms/step - accuracy: 0.5360 - auc: 0.5387 - loss: 0.6924 - val_accuracy: 0.5836 - val_auc: 0.6625 - val_loss: 0.6915
Epoch 5/25
33/33 - 8s - 249ms/step - accuracy: 0.5303 - auc: 0.5622 - loss: 0.6888 - val_accuracy: 0.5581 - val_auc: 0.6318 - val_loss: 0.6846
Epoch 6/25
33/33 - 8s - 239ms/step - accuracy: 0.5530 - auc: 0.6062 - loss: 0.6771 - val_accuracy: 0.6034 - val_auc: 0.6461 - val_loss: 0.6714
Epoch 7/25
33/33 - 8s - 245ms/step - accuracy: 0.6032 - auc: 0.6497 - loss: 0.6570 - val_accuracy: 0.6289 - val_auc: 0.6628 - val_loss: 0.656

In [24]:
#Save everything
config=dict(mapping=MAPPING,class_weights=class_weights,seeds=[42,43,44],
    tensorflow_version=tf.__version__,split_fingerprint=fingerprint,
    threshold=.5,test_evaluated=False,max_epochs=25,patience=5,
    batch_size=32,steps_per_epoch=31,learning_rate=.001)
(out/'binary_baseline_config.json').write_text(json.dumps(config,indent=2))
print('ALL_BINARY_BASELINES_COMPLETE')
print(pd.DataFrame(results).to_string(index=False))
# No test image dataset is constructed or evaluated in this script.


ALL_BINARY_BASELINES_COMPLETE
 seed     condition  epochs_trained  selected_epoch  validation_loss  accuracy  macro_f1  balanced_accuracy  malignant_sensitivity  specificity  roc_auc  average_precision  threshold                                           initial_weights_sha256                                                split_fingerprint  test_evaluated
   42 binary_no_aug              25              25         0.523471  0.742210  0.741911           0.742840               0.787356     0.698324 0.829545           0.808583        0.5 58664884d2acf89bb923ccd2b745a10c5499e3998bd4ffa6ea96e534e8bc564c 0339f080771817630bd0420c8dbef55b650000ecb9001718d1ec84e296487af2           False
   43 binary_no_aug              25              24         0.543890  0.713881  0.710874           0.715469               0.827586     0.603352 0.812496           0.800544        0.5 fb2910c5637b9951cc5e54ac36003c23da39759070ebfd9d0517cf7b89f84e07 0339f080771817630bd0420c8dbef55b650000ecb9001718d1ec84e296487af2